# Pre-processing MMFT data (D3.4) for the 4-Growth Platform

London Economics reformatted the MMFT workbooks for D3.4. Every scenario sheet is now a single
flat table with one header row, and both the agriculture and forestry workbooks share the same
layout:

```
Scenario | Ouput | Technology | Technology subcategory | Technology type | Units | Region | Country | 2020 ... 2040
```

so the section splitting that `03_pre-process_scenario_data.ipynb` needs for the D3.3 format is no
longer required. That notebook is kept as-is in case the partners revert to the old structure.

This notebook writes the same eight CSVs, with the same column contract.

## Setup and Imports

### Import Libraries

In [1]:
from pathlib import Path

import pandas as pd

### Variables

In [2]:
# Local directory for raw data files (Excel sheets)
INPUT_DATA_DIR = Path("../data/raw/D3.4/")
OUTPUT_DATA_DIR = Path("../data/processed/D3.4/")

FORESTRY_MMFT_FILE = "Forestry D3.4 VZZ.xlsx"
AGRICULTURE_MMFT_FILE = "Agri_D3.4 VZZ v3.xlsx"

# Scenario sheet names. The two workbooks capitalise them differently, and the agriculture one also
# holds a hidden "Scenario parameters - old" sheet, so the sheets are always read by explicit name.
FORESTRY_MMFT_SHEET_NAMES = {
    "baseline": "Baseline",
    "reimagining_progress": "Reimagining progress",
    "fractured_continent": "Fractured continent",
    "corporate_epoch": "Corporate epoch",
}

AGRICULTURE_MMFT_SHEET_NAMES = {
    "baseline": "Baseline",
    "reimagining_progress": "Reimagining Progress",
    "fractured_continent": "Fractured Continent",
    "corporate_epoch": "Corporate Epoch",
}

# "Ouput" (sic) values mapped to the indicator names the platform expects. D3.4 dropped the
# "Market potential" output that D3.3 had, so only six indicators are produced.
OUTPUT_TO_INDICATOR = {
    "Addressable Market": "addressable_market",
    "Penetration": "penetration",
    "Shipments": "shipments",
    "Installed base": "installed_base",
    "Prices": "prices",
    "Revenue": "revenues",
}

# The platform resolves countries with an exact match against shared/constants/country-iso.map.ts,
# which spells this one "Netherlands". Without the rename its rows are dropped on import.
COUNTRY_RENAMES = {"The Netherlands": "Netherlands"}

# Dimension columns as they are named in the raw sheets
DIMENSION_SOURCE_COLUMNS = [
    "Technology",
    "Technology subcategory",
    "Technology type",
    "Units",
    "Region",
    "Country",
]

# Output column contract, matching the D3.3 CSVs that projections.parser.ts reads by position
YEAR_COLUMNS = [str(year) for year in range(2020, 2041)]
OUTPUT_COLUMNS = [
    "Tech #",
    "Technology",
    "Technology subcategory",
    "Technology type",
    "Unit",
    "Region",
    "Country",
    *YEAR_COLUMNS,
    "Indicator",
]

### Utility Functions

In [3]:
def to_platform_format(df_raw: pd.DataFrame, label: str) -> pd.DataFrame:
    """
    Reshape one raw D3.4 scenario sheet into the CSV contract the platform imports.

    Drops the blank spacer rows, then reports and removes rows that cannot be imported:
    rows with missing dimensions, and rows outside the EU Europe region (the platform only
    resolves individual countries, not the aggregate regions).

    Returns:
        DataFrame with the columns listed in OUTPUT_COLUMNS
    """
    df = df_raw.dropna(how="all")

    incomplete = df[df[DIMENSION_SOURCE_COLUMNS].isna().any(axis=1)]
    if not incomplete.empty:
        print(f"  {label}: dropping {len(incomplete)} row(s) with missing dimensions")
        print(incomplete[["Ouput", *DIMENSION_SOURCE_COLUMNS]].to_string())
        df = df.drop(index=incomplete.index)

    non_eu = df[df["Region"] != "EU Europe"]
    if not non_eu.empty:
        print(f"  {label}: dropping {len(non_eu)} row(s) outside EU Europe")
        print(non_eu[["Ouput", "Technology", "Region", "Country"]].to_string())
        df = df.drop(index=non_eu.index)

    unmapped = set(df["Ouput"]) - set(OUTPUT_TO_INDICATOR)
    if unmapped:
        raise ValueError(f"Unmapped output(s) in {label}: {sorted(unmapped)}")

    df = df.assign(
        **{
            "Indicator": df["Ouput"].map(OUTPUT_TO_INDICATOR),
            "Country": df["Country"].replace(COUNTRY_RENAMES),
            # Column 0 is ignored on import; it only keeps the file readable and legacy-shaped
            "Tech #": pd.factorize(df["Technology"])[0] + 1,
        }
    ).rename(columns={"Units": "Unit"})

    # Year headers come out of Excel as numbers, the contract expects them as plain strings
    df.columns = [str(int(col)) if isinstance(col, int | float) else col for col in df.columns]

    # Drops the redundant "Scenario" column: the scenario is carried by the file name
    return df.reindex(columns=OUTPUT_COLUMNS)

## Data Loading and Pre-processing

**Forestry MMFT Data**

In [4]:
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

forestry_data: dict[str, pd.DataFrame] = {}

for scenario, sheet_name in FORESTRY_MMFT_SHEET_NAMES.items():
    df_raw = pd.read_excel(INPUT_DATA_DIR / FORESTRY_MMFT_FILE, sheet_name=sheet_name)
    print(f"Loaded {sheet_name}: {len(df_raw)} rows")

    forestry_data[scenario] = to_platform_format(df_raw, f"forestry {scenario}")
    forestry_data[scenario].to_csv(OUTPUT_DATA_DIR / f"forestry_{scenario}.csv", index=False)
    print(f"  wrote forestry_{scenario}.csv: {len(forestry_data[scenario])} rows")

Loaded Baseline: 810 rows
  wrote forestry_baseline.csv: 810 rows
Loaded Reimagining progress: 810 rows
  wrote forestry_reimagining_progress.csv: 810 rows


Loaded Fractured continent: 810 rows
  wrote forestry_fractured_continent.csv: 810 rows
Loaded Corporate epoch: 811 rows
  forestry corporate_epoch: dropping 1 row(s) with missing dimensions
              Ouput Technology Technology subcategory Technology type Units Region Country
405  Installed base        NaN                    NaN             NaN   NaN    NaN     NaN
  forestry corporate_epoch: dropping 15 row(s) outside EU Europe
              Ouput                   Technology                Region               Country
432  Installed base    Field Survey Technologies  Africa & Middle East  Africa & Middle East
459  Installed base    Field Survey Technologies  Africa & Middle East  Africa & Middle East
486  Installed base  Remote Sensing Technologies  Africa & Middle East  Africa & Middle East
513  Installed base     Decision Support Systems  Africa & Middle East  Africa & Middle East
540  Installed base           Telematics Systems  Africa & Middle East  Africa & Middle East
567 

**Agriculture MMFT Data**

In [5]:
agriculture_data: dict[str, pd.DataFrame] = {}

for scenario, sheet_name in AGRICULTURE_MMFT_SHEET_NAMES.items():
    df_raw = pd.read_excel(INPUT_DATA_DIR / AGRICULTURE_MMFT_FILE, sheet_name=sheet_name)
    print(f"Loaded {sheet_name}: {len(df_raw)} rows")

    agriculture_data[scenario] = to_platform_format(df_raw, f"agriculture {scenario}")
    agriculture_data[scenario].to_csv(OUTPUT_DATA_DIR / f"agriculture_{scenario}.csv", index=False)
    print(f"  wrote agriculture_{scenario}.csv: {len(agriculture_data[scenario])} rows")

Loaded Baseline: 2916 rows
  wrote agriculture_baseline.csv: 2916 rows


Loaded Reimagining Progress: 2916 rows
  wrote agriculture_reimagining_progress.csv: 2916 rows


Loaded Fractured Continent: 2916 rows
  wrote agriculture_fractured_continent.csv: 2916 rows


Loaded Corporate Epoch: 2916 rows
  wrote agriculture_corporate_epoch.csv: 2916 rows


### Checks

Verifies that what was written still matches the contract `projections.parser.ts` reads, so a
future re-formatting by the partners fails here instead of silently dropping rows on import.

In [6]:
LEGACY_HEADER = pd.read_csv(
    Path("../data/processed/D3.3/agriculture_baseline.csv"), nrows=0
).columns.tolist()

EU_COUNTRY_COUNT = 27

for domain, data in (("forestry", forestry_data), ("agriculture", agriculture_data)):
    for scenario, df in data.items():
        label = f"{domain}_{scenario}"
        assert list(df.columns) == LEGACY_HEADER, f"{label}: header does not match the contract"
        assert set(df["Indicator"]) == set(OUTPUT_TO_INDICATOR.values()), f"{label}: indicators"
        assert set(df["Region"]) == {"EU Europe"}, f"{label}: unexpected region"
        assert df["Country"].nunique() == EU_COUNTRY_COUNT, f"{label}: unexpected country count"
        assert df[YEAR_COLUMNS].notna().all().all(), f"{label}: NaN in year columns"

        # The API scales penetration by 100 on import, so it has to arrive as a 0-1 fraction
        penetration = df.loc[df["Indicator"] == "penetration", YEAR_COLUMNS]
        assert penetration.max().max() <= 1, f"{label}: penetration is not a fraction"

        print(f"{label}: {len(df)} rows, {dict(df['Indicator'].value_counts())}")

forestry_baseline: 810 rows, {'addressable_market': np.int64(135), 'penetration': np.int64(135), 'shipments': np.int64(135), 'installed_base': np.int64(135), 'prices': np.int64(135), 'revenues': np.int64(135)}
forestry_reimagining_progress: 810 rows, {'addressable_market': np.int64(135), 'penetration': np.int64(135), 'shipments': np.int64(135), 'installed_base': np.int64(135), 'prices': np.int64(135), 'revenues': np.int64(135)}
forestry_fractured_continent: 810 rows, {'addressable_market': np.int64(135), 'penetration': np.int64(135), 'shipments': np.int64(135), 'installed_base': np.int64(135), 'prices': np.int64(135), 'revenues': np.int64(135)}
forestry_corporate_epoch: 795 rows, {'addressable_market': np.int64(135), 'penetration': np.int64(135), 'shipments': np.int64(135), 'installed_base': np.int64(130), 'prices': np.int64(130), 'revenues': np.int64(130)}
agriculture_baseline: 2916 rows, {'addressable_market': np.int64(486), 'penetration': np.int64(486), 'shipments': np.int64(486), '